# Hierarchischer Ansatz: LDA

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** LDA mit Shrinkage als Kultur- und Stadium-Modell im hierarchischen Ansatz testen (#98).

In [ ]:
%load_ext autoreload
%autoreload 2


from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.data import load_split
from awp2.evaluation import compare_predictions
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import HierarchicalClassifier, summarize_hierarchical, summary_table
from awp2.models.linear import BalancedLinearDiscriminantAnalysis
from awp2.models.tuned import refit_untracked, tuned_hierarchical_svm
from awp2.plots import plot_error_breakdown, plot_tuning_candidates, save_doc_figure
from awp2.preprocessing import PreprocessingConfig

In [ ]:
split = load_split()
split.X_train.shape

## Hypothese

Die Spektralbänder sind stark korreliert; LDA mit Shrinkage ist dafür ein Klassiker der Spektroskopie und braucht kaum Hyperparameter. Mit gleichen Priors je Klasse (`BalancedLinearDiscriminantAnalysis`) passt sie zur Balanced Accuracy. Frage: Reicht eine lineare Trennung je Stufe, oder braucht es die nichtlineare RBF-SVM?

Gleicher Ablauf wie bei der Referenz `hierarchical_svm` (#87): CV-Suche mit `tune()` auf dem
Trainingsteil, dann genau ein `run()`. Verglichen wird per CV mit der SVM-Referenz, danach
Fehlerzerlegung, Oracle-Kultur und McNemar gegen die SVM auf der Validierung.

In [ ]:
reference = tuned_hierarchical_svm()
svm_cv_score = reference.cv_score
preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
TOP_CANDIDATES = 15

model = HierarchicalClassifier(
    BalancedLinearDiscriminantAnalysis(solver="lsqr"),
    BalancedLinearDiscriminantAnalysis(solver="lsqr"),
)

shrinkage = ["auto", 0.01, 0.1, 0.5]
param_grid = {
    "crop_estimator__shrinkage": shrinkage,
    "stage_estimator__shrinkage": shrinkage,
}

## CV-Suche

In [ ]:
tuned = tune(
    model,
    TuneConfig(
        name="hierarchical_lda_search",
        approach="hierarchical",
        description="LDA je Stufe im hierarchischen Ansatz (#98).",
        preprocessing=preprocessing,
        balance_samples=False,
        param_grid=param_grid,
    ),
)
tuned.best_params, round(tuned.best_score, 4), round(tuned.best_score - svm_cv_score, 4)

In [ ]:
tuning_ax = plot_tuning_candidates(
    tuned.candidates.nsmallest(TOP_CANDIDATES, "rank"), color_by="crop_estimator__shrinkage"
)
tuning_ax.axvline(svm_cv_score, color="black", linestyle="--", linewidth=1)
tuning_ax.set_title("Hierarchisch LDA: CV-Kandidaten (gestrichelt: SVM-Referenz)")
save_doc_figure(tuning_ax.figure, "hierarchical_lda_tuning", MODEL_DOCS_FIGURES_DIR)

## Einmalige Bewertung

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="hierarchical_lda",
        approach="hierarchical",
        description="Bester LDA aus hierarchical_lda_search, einmal bewertet.",
        preprocessing=preprocessing,
        balance_samples=False,
        tuning_run=tuned.run_id,
    ),
)
result.metrics.model_dump(), result.run_id

## Vergleich mit der SVM-Referenz

Die SVM-Referenz (`tuned_hierarchical_svm()`) wird ohne MLflow nachtrainiert
(`refit_untracked`), um die Vorhersagen Zeile für Zeile zu vergleichen.

In [ ]:
svm_result = refit_untracked(reference, "hierarchical_svm_refit")
summaries = {
    "RBF-SVM (Referenz)": summarize_hierarchical(svm_result.pipeline, split.X_val, split.y_val),
    "LDA": summarize_hierarchical(result.pipeline, split.X_val, split.y_val),
}
breakdowns = {label: summary.breakdown for label, summary in summaries.items()}
(
    summary_table(summaries).round(3),
    compare_predictions(split.y_val, svm_result.y_pred, result.y_pred),
)

In [ ]:
breakdown_ax = plot_error_breakdown(breakdowns)
breakdown_ax.set_title("Fehlerzerlegung: LDA vs. RBF-SVM")
save_doc_figure(breakdown_ax.figure, "hierarchical_lda_breakdown", MODEL_DOCS_FIGURES_DIR)

## Befund

- LDA (gleiche Priors, Shrinkage) erreicht CV 0.657 und Validierung 0.676 – weit unter der
  RBF-SVM (0.839 / 0.844); McNemar 340 zu 45 Zeilen zugunsten der SVM.
- Der Verlust liegt vor allem in der **Kultur**: 23 % falsche Kultur statt 9 %. Mit wahrer
  Kultur erreicht LDA 0.901 (SVM 0.939) – innerhalb einer Kultur trennt eine lineare Grenze die
  Stadien recht gut, zwischen den Kulturen nicht.
- Das Stadium-Optimum liegt bei der kleinsten Shrinkage (0.01) am Rasterrand; die Kultur nutzt
  `auto`.